<a href="https://colab.research.google.com/github/CelalDemirsoy/Yolo_Test/blob/main/train_to_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics

In [ ]:
import os

# 1. Kaggle klasörü oluşturuyoruz
!mkdir -p ~/.kaggle

# 2. Yüklediğin dosyayı olması gereken yere taşıyoruz
!cp /content/kaggle.json ~/.kaggle/

# 3. Güvenlik izni veriyoruz (Hata almamak için şart)
!chmod 600 ~/.kaggle/kaggle.json

print("Kaggle bağlantısı başarıyla kuruldu!")

cp: cannot stat '/content/kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Kaggle bağlantısı başarıyla kuruldu!


In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split

# 1. DOĞRU VERİ SETİNİ İNDİRME (Ekran görüntündeki kullanıcı adı ile)
# Eğer '403 Forbidden' hatası alırsan, web sitesinden "Download"a bastığından emin ol.
!kaggle datasets download -d nguyngiabol/colorful-fashion-dataset-for-object-detection

# 2. ZIP'TEN ÇIKARMA
print("Dosya indirildi, açılıyor...")
!unzip -q colorful-fashion-dataset-for-object-detection.zip -d /content/ham_veri

# 3. KLASÖRLERİ YOLO FORMATINA DÖNÜŞTÜRME
print("Dosyalar ayrıştırılıyor (Train/Val)...")

# Kaynak klasörler (İndirilen verinin içindeki yollar)
source_images = '/content/ham_veri/JPEGImages'
source_labels = '/content/ham_veri/Annotations_txt' # YOLO formatı burada

# Hedef klasör (YOLO'nun çalışacağı yer)
base_dir = '/content/dataset'

# Klasörleri oluştur
for folder in ['train/images', 'train/labels', 'val/images', 'val/labels']:
    os.makedirs(os.path.join(base_dir, folder), exist_ok=True)

# Resim dosyalarını bul
image_files = [f for f in os.listdir(source_images) if f.endswith('.jpg') or f.endswith('.png')]

# Veriyi %85 Eğitim, %15 Test olarak ayır
train_files, val_files = train_test_split(image_files, test_size=0.15, random_state=42)

def move_files(files, split):
    for filename in files:
        name_no_ext = os.path.splitext(filename)[0]

        # Resim ve Etiket yolları
        src_img = os.path.join(source_images, filename)
        src_txt = os.path.join(source_labels, name_no_ext + '.txt')

        # Eğer etiketi varsa taşı
        if os.path.exists(src_txt):
            shutil.copy(src_img, os.path.join(base_dir, split, 'images', filename))
            shutil.copy(src_txt, os.path.join(base_dir, split, 'labels', name_no_ext + '.txt'))

move_files(train_files, 'train')
move_files(val_files, 'val')

print("✅ MÜKEMMEL! Veri seti başarıyla indirildi ve ayrıştırıldı.")
print(f"Eğitim seti sayısı: {len(os.listdir('/content/dataset/train/images'))}");

Traceback (most recent call last):
  File "/usr/local/bin/kaggle", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/cli.py", line 68, in main
    out = args.func(**command_args)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 1741, in dataset_download_cli
    with self.build_kaggle_client() as kaggle:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 688, in build_kaggle_client
    username=self.config_values['username'],
             ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'username'
Dosya indirildi, açılıyor...
unzip:  cannot find or open colorful-fashion-dataset-for-object-detection.zip, colorful-fashion-dataset-for-object-detection.zip.zip or colorful-fashion-dataset-for-object-detection.zip.ZIP.
Dosyalar ayrıştırılıyor (Train/Val)...


FileNotFoundError: [Errno 2] No such file or directory: '/content/ham_veri/JPEGImages'

In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split

# 1. Klasörlerin Gerçek Yerini Bulalım
print("Klasörler aranıyor...")
search_dir = '/content/ham_veri'
found_images = None
found_labels = None

# Tüm alt klasörleri gezip doğru yeri buluyoruz
for root, dirs, files in os.walk(search_dir):
    if 'JPEGImages' in dirs:
        found_images = os.path.join(root, 'JPEGImages')
    if 'Annotations_txt' in dirs:
        found_labels = os.path.join(root, 'Annotations_txt')

if not found_images or not found_labels:
    print("HATA: Klasörler bulunamadı! Lütfen dosya yapısını kontrol et.")
else:
    print(f"BULDUM!\nResimler burada: {found_images}\nEtiketler burada: {found_labels}")

    # 2. YOLO Klasörlerini Hazırla
    base_dir = '/content/dataset'
    # Eski dataset varsa temizleyelim ki karışmasın
    if os.path.exists(base_dir):
        shutil.rmtree(base_dir)

    for folder in ['train/images', 'train/labels', 'val/images', 'val/labels']:
        os.makedirs(os.path.join(base_dir, folder), exist_ok=True)

    # 3. Dosyaları Ayır ve Taşı
    print("Dosyalar ayrıştırılıyor...")
    image_files = [f for f in os.listdir(found_images) if f.endswith('.jpg') or f.endswith('.png')]

    # %85 Eğitim, %15 Doğrulama
    train_files, val_files = train_test_split(image_files, test_size=0.15, random_state=42)

    def move_files(file_list, split_name):
        count = 0
        for filename in file_list:
            name_no_ext = os.path.splitext(filename)[0]

            src_img = os.path.join(found_images, filename)
            src_txt = os.path.join(found_labels, name_no_ext + '.txt')

            # Eğer txt dosyası varsa taşı
            if os.path.exists(src_txt):
                shutil.copy(src_img, os.path.join(base_dir, split_name, 'images', filename))
                shutil.copy(src_txt, os.path.join(base_dir, split_name, 'labels', name_no_ext + '.txt'))
                count += 1
        return count

    train_count = move_files(train_files, 'train')
    val_count = move_files(val_files, 'val')

    print(f"\n✅ İŞLEM TAMAMLANDI!")
    print(f"Eğitim için {train_count} resim hazırlandı.")
    print(f"Test için {val_count} resim hazırlandı.")

Klasörler aranıyor...
HATA: Klasörler bulunamadı! Lütfen dosya yapısını kontrol et.


In [ ]:
# Veri setinin haritasını (YAML dosyasını) oluşturuyoruz
yaml_content = """
path: /content/dataset  # Ana klasör
train: train/images     # Eğitim resimleri
val: val/images         # Test resimleri

# Sınıf Sayısı (Bu veri setinde 10 çeşit nesne var)
nc: 10

# Sınıf İsimleri (Sıralama çok önemlidir)
names:
  0: 'Gunes Gozlugu'
  1: 'Sapka'
  2: 'Ceket'
  3: 'Gomlek'
  4: 'Pantolon'
  5: 'Sort'
  6: 'Etek'
  7: 'Elbise'
  8: 'Canta'
  9: 'Ayakkabi'
"""

with open('/content/dataset/data.yaml', 'w') as f:
    f.write(yaml_content)

print("✅ data.yaml dosyası oluşturuldu! Eğitime hazırız.")

✅ data.yaml dosyası oluşturuldu! Eğitime hazırız.


In [ ]:
from ultralytics import YOLO

# 1. Modeli Yükle (Transfer Learning)
# Daha önce milyonlarca resim görmüş 'nano' modeli çağırıyoruz.
model = YOLO("yolov8n.pt")

print("Eğitim başlıyor... Bu işlem 15-20 dakika sürebilir.")

# 2. Eğitimi Başlat
results = model.train(
    data="/content/dataset/data.yaml",  # Hazırladığımız harita
    epochs=30,                          # Verinin üzerinden 30 kere geç
    imgsz=640,                          # Resim boyutu
    batch=16,                           # Aynı anda 16 resme bak
    name='moda_modelim'                 # Sonuçları bu klasöre kaydet
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Eğitim başlıyor... Bu işlem 15-20 dakika sürebilir.
Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, 

FileNotFoundError: [34m[1mtrain: [0mError loading data from /content/dataset/train/images
See https://docs.ultralytics.com/datasets for dataset formatting guidance.

In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

# ---------------- ADIM 1: TEMİZLİK VE HAZIRLIK ----------------
print("🚀 Sistem hazırlanıyor...")

# Daha önce yarım kalan dosyalar varsa temizleyelim
if os.path.exists('/content/dataset'):
    shutil.rmtree('/content/dataset')
if os.path.exists('/content/ham_veri'):
    shutil.rmtree('/content/ham_veri')

# ---------------- ADIM 2: VERİYİ İNDİRME ----------------
print("📦 Veri seti indiriliyor...")
# Eğer kaggle.json yoksa hata verir, lütfen yüklediğinden emin ol
if not os.path.exists('/content/kaggle.json') and not os.path.exists('/root/.kaggle/kaggle.json'):
    print("❌ HATA: 'kaggle.json' dosyası bulunamadı! Lütfen dosya menüsüne yükleyin.")
else:
    # İzinleri ayarla
    os.system('mkdir -p ~/.kaggle')
    os.system('cp /content/kaggle.json ~/.kaggle/')
    os.system('chmod 600 ~/.kaggle/kaggle.json')

    # İndir ve Çıkar
    os.system('kaggle datasets download -d nguyngiabol/colorful-fashion-dataset-for-object-detection')
    os.system('unzip -q colorful-fashion-dataset-for-object-detection.zip -d /content/ham_veri')

# ---------------- ADIM 3: KLASÖRLERİ BULMA VE AYIRMA ----------------
print("📂 Dosyalar düzenleniyor...")

# Zip'in içindeki gerçek klasörleri bul
search_dir = '/content/ham_veri'
found_images = None
found_labels = None

for root, dirs, files in os.walk(search_dir):
    if 'JPEGImages' in dirs:
        found_images = os.path.join(root, 'JPEGImages')
    if 'Annotations_txt' in dirs:
        found_labels = os.path.join(root, 'Annotations_txt')

if not found_images or not found_labels:
    print("❌ HATA: İndirilen veride klasörler bulunamadı. Lütfen kaggle.json'ı kontrol edip tekrar dene.")
else:
    # Hedef klasörleri oluştur
    base_dir = '/content/dataset'
    for folder in ['train/images', 'train/labels', 'val/images', 'val/labels']:
        os.makedirs(os.path.join(base_dir, folder), exist_ok=True)

    # Dosyaları listele
    image_files = [f for f in os.listdir(found_images) if f.endswith('.jpg') or f.endswith('.png')]

    # %85 Eğitim, %15 Test olarak ayır
    train_files, val_files = train_test_split(image_files, test_size=0.15, random_state=42)

    def move_files(files, split):
        count = 0
        for filename in files:
            name_no_ext = os.path.splitext(filename)[0]
            src_img = os.path.join(found_images, filename)
            src_txt = os.path.join(found_labels, name_no_ext + '.txt')

            if os.path.exists(src_txt):
                shutil.copy(src_img, os.path.join(base_dir, split, 'images', filename))
                shutil.copy(src_txt, os.path.join(base_dir, split, 'labels', name_no_ext + '.txt'))
                count += 1
        return count

    t_count = move_files(train_files, 'train')
    v_count = move_files(val_files, 'val')
    print(f"✅ Veri hazır! {t_count} eğitim, {v_count} test resmi ayrıldı.")

    # ---------------- ADIM 4: YAML OLUŞTURMA ----------------
    yaml_content = """
path: /content/dataset
train: train/images
val: val/images

nc: 10
names:
  0: 'Gunes Gozlugu'
  1: 'Sapka'
  2: 'Ceket'
  3: 'Gomlek'
  4: 'Pantolon'
  5: 'Sort'
  6: 'Etek'
  7: 'Elbise'
  8: 'Canta'
  9: 'Ayakkabi'
"""
    with open('/content/dataset/data.yaml', 'w') as f:
        f.write(yaml_content)

    # ---------------- ADIM 5: EĞİTİMİ BAŞLATMA ----------------
    print("🔥 EĞİTİM BAŞLIYOR! (Bu işlem 15-20 dk sürebilir)")

    model = YOLO("yolov8n.pt")
    results = model.train(
        data="/content/dataset/data.yaml",
        epochs=30,
        imgsz=640,
        batch=16,
        name='moda_modelim'
    )

🚀 Sistem hazırlanıyor...
📦 Veri seti indiriliyor...
📂 Dosyalar düzenleniyor...
✅ Veri hazır! 2279 eğitim, 403 test resmi ayrıldı.
🔥 EĞİTİM BAŞLIYOR! (Bu işlem 15-20 dk sürebilir)
Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.93

In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

# --- ADIM 0: KOD ÇALIŞMAYA BAŞLIYOR ---
print("🚀 PRO EĞİTİM MODU BAŞLATILIYOR...")

# Eski verileri temizleyelim (Çakışma olmasın)
if os.path.exists('/content/dataset'): shutil.rmtree('/content/dataset')
if os.path.exists('/content/ham_veri'): shutil.rmtree('/content/ham_veri')

# --- ADIM 1: VERİYİ İNDİR ---
print("📦 Veri seti indiriliyor...")
# Kaggle kimliği kontrolü
if not os.path.exists('/content/kaggle.json') and not os.path.exists('/root/.kaggle/kaggle.json'):
    print("❌ HATA: Lütfen 'kaggle.json' dosyasını sol menüye yükle!")
else:
    os.system('mkdir -p ~/.kaggle')
    os.system('cp /content/kaggle.json ~/.kaggle/')
    os.system('chmod 600 ~/.kaggle/kaggle.json')
    # Dataseti indir
    os.system('kaggle datasets download -d nguyngiabol/colorful-fashion-dataset-for-object-detection')
    os.system('unzip -q colorful-fashion-dataset-for-object-detection.zip -d /content/ham_veri')

# --- ADIM 2: KLASÖRLERİ BUL VE AYIR ---
print("📂 Dosyalar düzenleniyor...")
search_dir = '/content/ham_veri'
found_images = None
found_labels = None

for root, dirs, files in os.walk(search_dir):
    if 'JPEGImages' in dirs: found_images = os.path.join(root, 'JPEGImages')
    if 'Annotations_txt' in dirs: found_labels = os.path.join(root, 'Annotations_txt')

if found_images and found_labels:
    base_dir = '/content/dataset'
    for folder in ['train/images', 'train/labels', 'val/images', 'val/labels']:
        os.makedirs(os.path.join(base_dir, folder), exist_ok=True)

    image_files = [f for f in os.listdir(found_images) if f.endswith('.jpg') or f.endswith('.png')]
    # %85 Eğitim, %15 Test
    train_files, val_files = train_test_split(image_files, test_size=0.15, random_state=42)

    def move_files(files, split):
        for filename in files:
            name_no_ext = os.path.splitext(filename)[0]
            src_img = os.path.join(found_images, filename)
            src_txt = os.path.join(found_labels, name_no_ext + '.txt')
            if os.path.exists(src_txt):
                shutil.copy(src_img, os.path.join(base_dir, split, 'images', filename))
                shutil.copy(src_txt, os.path.join(base_dir, split, 'labels', name_no_ext + '.txt'))

    move_files(train_files, 'train')
    move_files(val_files, 'val')

    # --- ADIM 3: YAML DOSYASI ---
    yaml_content = """
path: /content/dataset
train: train/images
val: val/images
nc: 10
names:
  0: 'Gunes Gozlugu'
  1: 'Sapka'
  2: 'Ceket'
  3: 'Gomlek'
  4: 'Pantolon'
  5: 'Sort'
  6: 'Etek'
  7: 'Elbise'
  8: 'Canta'
  9: 'Ayakkabi'
"""
    with open('/content/dataset/data.yaml', 'w') as f:
        f.write(yaml_content)

    # --- ADIM 4: PRO EĞİTİM (YOLOv8 Small + 100 Epochs) ---
    print("🔥 EĞİTİM BAŞLIYOR! (Bu işlem uzun sürecektir...)")

    # DİKKAT: 'n' yerine 's' (Small) modelini yüklüyoruz.
    model = YOLO("yolov8s.pt")

    results = model.train(
        data="/content/dataset/data.yaml",
        epochs=100,         # 100 Tur (Daha iyi öğrenme)
        patience=15,        # 15 tur boyunca gelişmezse erken bitir (Vakit kazancı)
        imgsz=640,
        batch=16,           # Eğer hata alırsan bunu 8'e düşür
        name='moda_modelim_v2' # Yeni versiyon adı
    )
    print("✅ EĞİTİM TAMAMLANDI! Yeni modelini indirebilirsin.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🚀 PRO EĞİTİM MODU BAŞLATILIYOR...
📦 Veri seti indiriliyor...
📂 Dosyalar düzenleniyor...
🔥 EĞİTİM BAŞLIYOR! (Bu işlem uzun sürecektir...)
Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.

KeyboardInterrupt: 

In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

# --- ADIM 0: KOD ÇALIŞMAYA BAŞLIYOR ---
print("🚀 PRO EĞİTİM MODU BAŞLATILIYOR...")

# Eski verileri temizleyelim (Çakışma olmasın)
if os.path.exists('/content/dataset'): shutil.rmtree('/content/dataset')
if os.path.exists('/content/ham_veri'): shutil.rmtree('/content/ham_veri')

# --- ADIM 1: VERİYİ İNDİR ---
print("📦 Veri seti indiriliyor...")
# Kaggle kimliği kontrolü
if not os.path.exists('/content/kaggle.json') and not os.path.exists('/root/.kaggle/kaggle.json'):
    print("❌ HATA: Lütfen 'kaggle.json' dosyasını sol menüye yükle!")
else:
    os.system('mkdir -p ~/.kaggle')
    os.system('cp /content/kaggle.json ~/.kaggle/')
    os.system('chmod 600 ~/.kaggle/kaggle.json')
    # Dataseti indir
    os.system('kaggle datasets download -d nguyngiabol/colorful-fashion-dataset-for-object-detection')
    os.system('unzip -q colorful-fashion-dataset-for-object-detection.zip -d /content/ham_veri')

# --- ADIM 2: KLASÖRLERİ BUL VE AYIR ---
print("📂 Dosyalar düzenleniyor...")
search_dir = '/content/ham_veri'
found_images = None
found_labels = None

for root, dirs, files in os.walk(search_dir):
    if 'JPEGImages' in dirs: found_images = os.path.join(root, 'JPEGImages')
    if 'Annotations_txt' in dirs: found_labels = os.path.join(root, 'Annotations_txt')

if found_images and found_labels:
    base_dir = '/content/dataset'
    for folder in ['train/images', 'train/labels', 'val/images', 'val/labels']:
        os.makedirs(os.path.join(base_dir, folder), exist_ok=True)

    image_files = [f for f in os.listdir(found_images) if f.endswith('.jpg') or f.endswith('.png')]
    # %85 Eğitim, %15 Test
    train_files, val_files = train_test_split(image_files, test_size=0.15, random_state=42)

    def move_files(files, split):
        for filename in files:
            name_no_ext = os.path.splitext(filename)[0]
            src_img = os.path.join(found_images, filename)
            src_txt = os.path.join(found_labels, name_no_ext + '.txt')
            if os.path.exists(src_txt):
                shutil.copy(src_img, os.path.join(base_dir, split, 'images', filename))
                shutil.copy(src_txt, os.path.join(base_dir, split, 'labels', name_no_ext + '.txt'))

    move_files(train_files, 'train')
    move_files(val_files, 'val')

    # --- ADIM 3: YAML DOSYASI ---
    yaml_content = """
path: /content/dataset
train: train/images
val: val/images
nc: 10
names:
  0: 'Gunes Gozlugu'
  1: 'Sapka'
  2: 'Ceket'
  3: 'Gomlek'
  4: 'Pantolon'
  5: 'Sort'
  6: 'Etek'
  7: 'Elbise'
  8: 'Canta'
  9: 'Ayakkabi'
"""
    with open('/content/dataset/data.yaml', 'w') as f:
        f.write(yaml_content)

    # --- ADIM 4: PRO EĞİTİM (YOLOv8 Small + 100 Epochs) ---
    print("🔥 EĞİTİM BAŞLIYOR! (Bu işlem uzun sürecektir...)")

    # DİKKAT: 'n' yerine 's' (Small) modelini yüklüyoruz.
    model = YOLO("yolo11s.pt")

    results = model.train(
        data="/content/dataset/data.yaml",
        epochs=100,         # 100 Tur (Daha iyi öğrenme)
        patience=15,        # 15 tur boyunca gelişmezse erken bitir (Vakit kazancı)
        imgsz=640,
        batch=16,           # Eğer hata alırsan bunu 8'e düşür
        name='moda_modelim_v2' # Yeni versiyon adı
    )
    print("✅ EĞİTİM TAMAMLANDI! Yeni modelini indirebilirsin.")

🚀 PRO EĞİTİM MODU BAŞLATILIYOR...
📦 Veri seti indiriliyor...
📂 Dosyalar düzenleniyor...
🔥 EĞİTİM BAŞLIYOR! (Bu işlem uzun sürecektir...)
Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=mo